In [1]:
import duckdb
import pandas

In [2]:
sql_query = '''
show tables
'''

with duckdb.connect('data/chicago_crimes.db') as con:
    display(con.sql(sql_query).df())

,name
0,chicago_crimes
1,gold_crimes_by_district
2,gold_crimes_by_month_type
3,silver_crimes
4,silver_crimes_type


In [5]:
import duckdb

CSV = "data/raw/chicago_crimes_2026.csv"

with duckdb.connect("data/chicago_crimes.db") as con:
    con.sql(f"""
        create or replace table chicago_crimes as
        select *
        from read_csv_auto('{CSV}', header = true)
    """)
    con.sql("select * as filas from chicago_crimes").show()

┌──────────┬─────────────┬─────────────────────┬───┬──────────────┬───────────────┬──────────────────────┐
│    ID    │ Case Number │        Date         │ … │   Latitude   │   Longitude   │       Location       │
│  int64   │   varchar   │      timestamp      │   │    double    │    double     │       varchar        │
├──────────┼─────────────┼─────────────────────┼───┼──────────────┼───────────────┼──────────────────────┤
│ 14351522 │ JK442283    │ 2026-10-06 00:00:00 │ … │ 41.887481203 │ -87.645372811 │ POINT (-87.6453728…  │
│ 14350574 │ JK440996    │ 2026-10-06 00:00:00 │ … │ 41.966047906 │ -87.661060061 │ POINT (-87.6610600…  │
│ 14354393 │ JK445985    │ 2026-10-06 00:00:00 │ … │ 41.929417949 │  -87.70612448 │ POINT (-87.7061244…  │
│ 14349985 │ JK440553    │ 2026-10-06 00:00:00 │ … │ 41.892017399 │ -87.624165306 │ POINT (-87.6241653…  │
│ 14350659 │ JK441321    │ 2026-10-06 00:00:00 │ … │ 41.692833841 │  -87.60431945 │ POINT (-87.6043194…  │
│ 14349733 │ JK440247    │ 2026-10-06

In [6]:
with duckdb.connect('/workspace/data/chicago_crimes.db', read_only=True) as con:
    df = con.sql("select * from gold_crimes_by_month_type order by 1, 3 desc").df()

df.head()

,month,primary_type,total_crimes,total_arrests,arrest_rate
0,2026-01-01,THEFT,3701,385.0,0.104
1,2026-01-01,BATTERY,3027,651.0,0.215
2,2026-01-01,CRIMINAL DAMAGE,1820,80.0,0.044
3,2026-01-01,ASSAULT,1438,204.0,0.142
4,2026-01-01,MOTOR VEHICLE THEFT,1334,44.0,0.033


In [7]:
con = duckdb.connect('/workspace/data/chicago_crimes.db', read_only=True)
con.sql("show all tables").show()
con.sql("describe chicago_crimes").show()

┌────────────────┬─────────┬──────────────────────┬──────────────────────┬─────────────────────────────────┬───────────┐
│    database    │ schema  │         name         │     column_names     │          column_types           │ temporary │
│    varchar     │ varchar │       varchar        │      varchar[]       │            varchar[]            │  boolean  │
├────────────────┼─────────┼──────────────────────┼──────────────────────┼─────────────────────────────────┼───────────┤
│ chicago_crimes │ main    │ chicago_crimes       │ [ID, Case Number, …  │ [BIGINT, VARCHAR, TIMESTAMP, …  │ false     │
│ chicago_crimes │ main    │ gold_crimes_by_dis…  │ [district, total_c…  │ [VARCHAR, BIGINT, INTEGER, DO…  │ false     │
│ chicago_crimes │ main    │ gold_crimes_by_mon…  │ [month, primary_ty…  │ [DATE, VARCHAR, BIGINT, HUGEI…  │ false     │
│ chicago_crimes │ main    │ silver_crimes        │ [crime_id, case_nu…  │ [BIGINT, VARCHAR, TIMESTAMP, …  │ false     │
│ chicago_crimes │ main    │ sil

In [9]:
with duckdb.connect('/workspace/data/chicago_crimes.db', read_only=True) as con:
    df = con.sql("""
        select * from gold_crimes_by_month_type
        order by month, total_crimes desc
    """).df()

df.head(100)

,month,primary_type,total_crimes,total_arrests,arrest_rate
0,2026-01-01,THEFT,3701,385.0,0.104
1,2026-01-01,BATTERY,3027,651.0,0.215
2,2026-01-01,CRIMINAL DAMAGE,1820,80.0,0.044
3,2026-01-01,ASSAULT,1438,204.0,0.142
4,2026-01-01,MOTOR VEHICLE THEFT,1334,44.0,0.033
...,...,...,...,...,...
95,2026-04-01,ROBBERY,327,29.0,0.089
96,2026-04-01,OFFENSE INVOLVING CHILDREN,147,9.0,0.061
97,2026-04-01,SEX OFFENSE,139,9.0,0.065
98,2026-04-01,CRIMINAL SEXUAL ASSAULT,133,4.0,0.030


In [10]:
with duckdb.connect('/workspace/data/chicago_crimes.db', read_only=True) as con:
    df = con.sql("""
        select * from silver_crimes_type
    """).df()

df.head(100)

,iucr,primary_type,description,fbi_code
0,0486,BATTERY,DOMESTIC BATTERY SIMPLE,08B
1,1152,DECEPTIVE PRACTICE,ILLEGAL USE CASH CARD,11
2,0920,MOTOR VEHICLE THEFT,ATTEMPT - AUTOMOBILE,07
3,2025,NARCOTICS,POSSESS - HALLUCINOGENS,18
4,2870,PUBLIC PEACE VIOLATION,PEEPING TOM,24
...,...,...,...,...
95,1199,DECEPTIVE PRACTICE,"PUBLIC AIR WIRE AND MAIL FRAUD - VIA WIRE, RAD...",11
96,2896,PUBLIC PEACE VIOLATION,TIRE DEFLATION DEVICE DEPLOYMENT,24
97,1791,OFFENSE INVOLVING CHILDREN,HARBOR RUNAWAY,20
98,2160,NARCOTICS,SALE / DELIVER - DRUG PARAPHERNALIA,26
